# Neo4j Homework — Graph Data Modelling

**Course:** Data Engineering, Fall 2026 · **Session 6:** Semi-structured data

This homework has two parts:

| Part | Topic | Expected time |
| --- | --- | --- |
| 1 | Model a relational schema as a property graph, and build it | ~1h 35m |
| 2 | Query the movie knowledge graph | ~1h 25m |
| | **Total** | **~3h** |

The times are what a solid solution should take if you have followed the
practice session — not a limit. If something takes you much longer, say so in
the final question; that is useful feedback, not a problem.

Work directly in this notebook: replace each `YOUR CYPHER HERE` with a working
query and run the cell so the output is saved. Submit the notebook **with its
outputs** — a cell with no output cannot be graded.

---

## Setup

The graph must be loaded before you start. From the `05_Neo4J` folder on your host:

```bash
docker compose up -d
docker exec -i neo4j cypher-shell -u neo4j -p password < sample_data/load.cypher
```

In [ ]:
import os
import pandas as pd
from neo4j import GraphDatabase

URI      = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
USER     = os.getenv('NEO4J_USER', 'neo4j')
PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print(f'Connected to {URI}')


def run(cypher, **params):
    """Run a Cypher query and return the rows as a pandas DataFrame."""
    with driver.session() as session:
        return pd.DataFrame([r.data() for r in session.run(cypher, **params)])


def run_raw(cypher, **params):
    """Run a write query and return its summary counters."""
    with driver.session() as session:
        return session.run(cypher, **params).consume().counters

Check the graph is loaded. If this is empty, run the load command above.

In [ ]:
run('MATCH (n) RETURN labels(n)[0] AS label, count(*) AS nodes ORDER BY label')

---
# Part 1 — From Tables to a Graph (~1h 35m)

## 1.1 Why graphs? (~10 min)

Watch this short talk on graph databases versus relational databases:

In [ ]:
from IPython.display import IFrame
IFrame('https://www.youtube.com/embed/NO3C-CWykkY', width=640, height=360)

**Question.** Name **three** things a graph database makes easier than a
relational one, and **one** thing a relational database still does better.
Answer in two or three sentences — do not just list keywords.

---

<font color='red'>**Your answer:**</font>

*(write here)*

## 1.2 Model the schema (~25 min)

Below is a small relational schema for a film production company.

```
actor(actor_id, name, birth_year, nationality)
film(film_id, title, release_year, budget)
studio(studio_id, name, city)

role(actor_id, film_id, character_name)      -- junction table
produced_by(film_id, studio_id)              -- junction table
```

**Task.** Write down the property-graph model you would use:

1. Which **node labels** do you create, and which properties go on each?
2. Which **relationship types**, in which **direction**?
3. Which property does **not** belong on any node, and why? (Look at `role`.)
4. Which of the five tables **disappear entirely** in the graph model?

A diagram is welcome but not required — a clear written description is enough.

---

<font color='red'>**Your answer:**</font>

*(write here)*

<details>
<summary>💡 Hint (open only if stuck)</summary>

Junction tables exist in the relational model purely to express a many-to-many
link. In a graph the link *is* the relationship, so a junction table with no
columns of its own has no counterpart. A junction table **with** extra columns
is different — those columns become relationship properties.
</details>

## 1.3 Build your model (~40 min)

Create the graph you just described, using this data. Use a **separate label
namespace** (`:HwActor`, `:HwFilm`, `:HwStudio`) so your homework graph cannot
collide with the movie graph used in Part 2.

**actors**

| actor_id | name | birth_year | nationality |
| --- | --- | --- | --- |
| 1 | Charlie Sheen | 1965 | US |
| 2 | Michael Douglas | 1944 | US |
| 3 | Martin Sheen | 1940 | US |
| 4 | Morgan Freeman | 1937 | US |

**films**

| film_id | title | release_year | budget |
| --- | --- | --- | --- |
| 1 | Wall Street | 1987 | 15000000 |
| 2 | The American President | 1995 | 62000000 |
| 3 | The Shawshank Redemption | 1994 | 25000000 |

**studios**

| studio_id | name | city |
| --- | --- | --- |
| 1 | Twentieth Century Fox | Los Angeles |
| 2 | Castle Rock | Beverly Hills |

**roles** (actor_id, film_id, character_name)

`(1, 1, 'Bud Fox')`, `(2, 1, 'Gordon Gekko')`, `(3, 1, 'Carl Fox')`,
`(3, 2, 'A.J. MacInerney')`, `(4, 3, 'Ellis Boyd Redding')`

**produced_by** (film_id, studio_id)

`(1, 1)`, `(2, 2)`, `(3, 2)`

### (a) Create the nodes (~15 min)

*Hint:* `CREATE` accepts several comma-separated patterns in one statement.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run_raw('''
#
# ''')

### (b) Create the relationships (~15 min)

Remember that `character_name` is a property of the **relationship**, not of the
actor or the film.

*Hint:* `MATCH` both endpoints first, then `MERGE` the relationship between them.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run_raw('''
#
# ''')

### (c) Verify (~10 min)

Return every actor, the character they played, and the film title — one row per
role, ordered by film then character.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

<details>
<summary>💡 Show solution</summary>

```python
# (a) Nodes
run_raw('''
CREATE (:HwActor {actor_id: 1, name: 'Charlie Sheen',   birth_year: 1965, nationality: 'US'}),
       (:HwActor {actor_id: 2, name: 'Michael Douglas', birth_year: 1944, nationality: 'US'}),
       (:HwActor {actor_id: 3, name: 'Martin Sheen',    birth_year: 1940, nationality: 'US'}),
       (:HwActor {actor_id: 4, name: 'Morgan Freeman',  birth_year: 1937, nationality: 'US'}),
       (:HwFilm {film_id: 1, title: 'Wall Street',              release_year: 1987, budget: 15000000}),
       (:HwFilm {film_id: 2, title: 'The American President',   release_year: 1995, budget: 62000000}),
       (:HwFilm {film_id: 3, title: 'The Shawshank Redemption', release_year: 1994, budget: 25000000}),
       (:HwStudio {studio_id: 1, name: 'Twentieth Century Fox', city: 'Los Angeles'}),
       (:HwStudio {studio_id: 2, name: 'Castle Rock',           city: 'Beverly Hills'})
''')

# (b) Relationships -- character_name is a RELATIONSHIP property
run_raw('''
UNWIND [[1, 1, 'Bud Fox'], [2, 1, 'Gordon Gekko'], [3, 1, 'Carl Fox'],
        [3, 2, 'A.J. MacInerney'], [4, 3, 'Ellis Boyd Redding']] AS row
MATCH (a:HwActor {actor_id: row[0]})
MATCH (f:HwFilm  {film_id:  row[1]})
MERGE (a)-[:ACTED_IN {character_name: row[2]}]->(f)
''')

run_raw('''
UNWIND [[1, 1], [2, 2], [3, 2]] AS row
MATCH (f:HwFilm   {film_id:   row[0]})
MATCH (s:HwStudio {studio_id: row[1]})
MERGE (f)-[:PRODUCED_BY]->(s)
''')

# (c) Verify
run('''
MATCH (a:HwActor)-[r:ACTED_IN]->(f:HwFilm)
RETURN a.name AS actor, r.character_name AS character, f.title AS film
ORDER BY film, character
''')
```

`UNWIND` turns a list into rows, which keeps the load to one statement per
relationship type instead of one per row.
</details>

## 1.4 Query your model (~20 min)

Answer each with a single Cypher query against your `:Hw*` graph.

### (a) Which actors appeared in a film produced by Castle Rock? (~6 min)

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

### (b) Martin Sheen and Charlie Sheen share a film. Which one, and what did each play? (~8 min)

*Hint:* match both actors into the same film node.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

### (c) How many films did each studio produce, and what is their total budget? (~6 min)

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

<details>
<summary>💡 Show solution</summary>

```python
# (a)
run('''
MATCH (a:HwActor)-[:ACTED_IN]->(:HwFilm)-[:PRODUCED_BY]->(:HwStudio {name: 'Castle Rock'})
RETURN DISTINCT a.name AS actor ORDER BY actor
''')

# (b) both actors into the same film
run('''
MATCH (a1:HwActor {name: 'Martin Sheen'})-[r1:ACTED_IN]->(f:HwFilm)
      <-[r2:ACTED_IN]-(a2:HwActor {name: 'Charlie Sheen'})
RETURN f.title AS film, r1.character_name AS martinPlayed, r2.character_name AS charliePlayed
''')

# (c)
run('''
MATCH (f:HwFilm)-[:PRODUCED_BY]->(s:HwStudio)
RETURN s.name AS studio, count(f) AS films, sum(f.budget) AS totalBudget
ORDER BY films DESC, studio
''')
```
</details>

### Clean up Part 1

Run this when you are finished with Part 1. `DETACH DELETE` removes the nodes
together with their relationships — a plain `DELETE` on a connected node is an
error, because Neo4j refuses to leave dangling relationships.

The label namespace is what makes this safe: it touches only your homework nodes.

In [ ]:
run_raw('MATCH (n) WHERE n:HwActor OR n:HwFilm OR n:HwStudio DETACH DELETE n')

---
# Part 2 — Querying the Movie Graph (~1h 25m)

These questions use the graph loaded from `sample_data/`:

```
(:Person)-[:ACTED_IN {roles}]->(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]-------->(:Movie)
(:User)-[:RATED {rating, rated_year}]->(:Movie)
(:User)-[:FOLLOWS {since}]->(:User)
```

Confirm it is loaded before continuing.

In [ ]:
run('MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n ORDER BY rel')

## 2.1 Basics (~15 min)

### (a) List the distinct genres in the database, alphabetically. (~3 min)

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

### (b) How many movies are in the database? (~3 min)

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

### (c) Titles of the movies released in 2016, with their runtime. (~4 min)

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

### (d) The five longest movies, longest first. (~5 min)

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

<details>
<summary>💡 Show solution</summary>

```python
# (a)
run('MATCH (g:Genre) RETURN DISTINCT g.name AS genre ORDER BY genre')

# (b)
run('MATCH (m:Movie) RETURN count(m) AS movies')

# (c)
run('''
MATCH (m:Movie) WHERE m.released = 2016
RETURN m.title AS title, m.runtime AS runtime ORDER BY title
''')

# (d)
run('MATCH (m:Movie) RETURN m.title, m.runtime ORDER BY m.runtime DESC LIMIT 5')
```
</details>

## 2.2 Relationships and aggregation (~35 min)

### (a) Number of directors per movie, most first. Include movies with none. (~10 min)

*Hint:* `OPTIONAL MATCH`, and `count` ignores nulls.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

### (b) People who directed a film they also acted in — name, film, and role. (~8 min)

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

### (c) Genres of the movies Tom Hanks acted in, with how many of his films fall in each. (~8 min)

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

### (d) Title and rating of every movie user 3 rated, highest first. (~8 min)

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

<details>
<summary>💡 Show solution</summary>

```python
# (a) OPTIONAL MATCH keeps movies with no director; count(d) skips the nulls
run('''
MATCH (m:Movie)
OPTIONAL MATCH (m)<-[:DIRECTED]-(d:Person)
RETURN m.title AS movie, count(d) AS directors
ORDER BY directors DESC, movie
''')

# (b) the same person on both ends of the pattern
run('''
MATCH (p:Person)-[r:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(p)
RETURN p.name AS person, m.title AS film, r.roles AS role ORDER BY person
''')

# (c)
run('''
MATCH (:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)-[:IN_GENRE]->(g:Genre)
RETURN g.name AS genre, count(m) AS films ORDER BY films DESC, genre
''')

# (d)
run('''
MATCH (:User {user_id: 3})-[r:RATED]->(m:Movie)
RETURN m.title AS movie, r.rating AS rating, r.rated_year AS year
ORDER BY rating DESC, movie
''')
```
</details>

## 2.3 Traversals (~35 min)

### (a) Top five movies by average rating, among those rated by at least 10 users. (~10 min)

*Hint:* you cannot filter an aggregate in `WHERE` directly — use `WITH` first.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

### (b) Shortest co-acting path between two people of your choice. (~8 min)

Return the names along the path. Pick two people who are actually connected —
check with a short query first if you need to.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

### (c) Recommend movies for user 3 using collaborative filtering. (~17 min)

Find users who rated at least **three** of the same movies as user 3, then
recommend the movies *those* users rated **4 or higher** that user 3 has **not**
rated. Rank by how many of them endorsed each movie.

*Hint:* three stages piped with `WITH` — find the neighbours, filter by overlap,
then collect what they liked.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

<details>
<summary>💡 Show solution</summary>

```python
# (a)
run('''
MATCH (:User)-[r:RATED]->(m:Movie)
WITH m, avg(r.rating) AS avgRating, count(r) AS votes
WHERE votes >= 10
RETURN m.title AS movie, round(avgRating, 2) AS avgRating, votes
ORDER BY avgRating DESC LIMIT 5
''')

# (b)
run('''
MATCH path = shortestPath(
  (:Person {name: 'Tom Hanks'})-[:ACTED_IN*..6]-(:Person {name: 'Keanu Reeves'})
)
RETURN [n IN nodes(path) | coalesce(n.name, n.title)] AS hops, length(path) AS hopCount
''')

# (c)
run('''
MATCH (me:User {user_id: 3})-[:RATED]->(m:Movie)<-[:RATED]-(other:User)
WITH me, other, count(m) AS overlap
WHERE overlap >= 3
MATCH (other)-[r:RATED]->(rec:Movie)
WHERE NOT (me)-[:RATED]->(rec) AND r.rating >= 4
RETURN rec.title AS recommendation,
       count(*) AS endorsements,
       round(avg(r.rating), 2) AS avgRating
ORDER BY endorsements DESC, avgRating DESC LIMIT 10
''')
```

Part (c) is the query that justifies a graph database: in SQL it needs a
self-join on the ratings table plus a `NOT EXISTS` subquery.
</details>

# Closure

In [ ]:
driver.close()
print('Connection closed')

Shut the environment down from the host:

```bash
docker compose down -v
```

---

**<center><font color='red'>Thank you for your effort!</font></center>**